In [ ]:
import pandas as pd
import numpy as np

import allel
from Bio import SeqIO
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

from matplotlib import pyplot as plt
from matplotlib.patches import FancyArrow
from matplotlib import cm
import seaborn as sns

from progressbar import ProgressBar
import itertools
import mathieu as mh

In [ ]:
base_dir = '/home/mathieu/postdoc_heasley/wine_subclade_spores/'
#fig_path = f'{base_dir}fig/'
fig_path = '/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'


strains_info = pd.read_csv(f'{base_dir}script/strains_info2.csv')
strains_info.index = strains_info['strain'].values

collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx')
collection.index = collection['ID'].values
wine_subclade = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]
strain_order = list(collection.loc[wine_subclade].sort_values(by='Y\' element')['ID'].values)
strain_yprime_color = dict(zip(strain_order,
                               [cm.viridis(i) for i in np.linspace(0,1,10)]))

In [ ]:
strains_info.loc[(strains_info['background']!='YJM434') & (strains_info['ho']!='WT') & (~strains_info['Missing'])].to_csv(f'{base_dir}script/strains_info_spores.csv', index=False)

In [ ]:
chr_list = pd.read_csv(f'{base_dir}data/ref/S288C.chr_list.tsv', sep='\t')
chr_list['Alias'] = [f'chr{i}' for i in range(1,17)] + ['chrMT']
chr_ctg_color = dict(zip(chr_list['contig_name'], np.tile(['0.4', '0.7'], 8)))
chr_ctg_to_alias = chr_list.set_index('contig_name')['alias'].to_dict()
chr_ctg_to_Alias = chr_list.set_index('contig_name')['Alias'].to_dict()
chr_Alias_to_ctg = {j:i for (i,j) in chr_ctg_to_Alias.items()}
chr_alias_to_ctg = {j:i for (i,j) in chr_ctg_to_alias.items()}

snpeff_conversion = pd.read_csv(f'{base_dir}data/ref/snpEff_chrom_name_conversion.txt', sep=' ', header=None)
snpeff_to_ctg = dict(zip(snpeff_conversion[1], snpeff_conversion[0]))

gt_order = ['0/1', '1/0', '0|1', '1|0', '1/1', '1|1']

# Import ref genome
with open(f'{base_dir}data/ref/S288C.fasta') as fi:
    ref_genome = [seq for seq in SeqIO.parse(fi, 'fasta')]

nuclear_chromosomes = [seq.id for seq in ref_genome][:-1]

# genome display variables
chrom_order = list(chr_list['contig_name'])
ref_chr_len = [len(seq.seq) for seq in ref_genome]
ref_chr_len_dict = dict(zip(chrom_order, ref_chr_len))

tig_off = pd.Series(ref_chr_len_dict, name='len')
tig_off = pd.concat([tig_off, tig_off.cumsum().rename('cumul_end')], axis=1)
tig_off['cumul_start'] = np.concatenate([[0], tig_off['len'].iloc[:-1].cumsum().values])

tig_off['cumul_start_plot'] = tig_off['cumul_start'] + np.cumsum(np.repeat(1e5, 17))

# Genetic distances in cM, taken from https://wiki.yeastgenome.org/index.php/Combined_Physical_and_Genetic_Maps_of_S._cerevisiae
genetic_distance = dict(zip(chrom_order[:16], [0.45, 0.30, 0.48, 0.31, 0.35, 0.48, 0.37, 0.30, 0.45, 0.3, 0.38, 0.36, 0.34, 0.37, 0.33, 0.29]))

In [ ]:
with open(f'{base_dir}data/ref/S288C_masked.repeats.fasta') as fi:
    ref_genome_masked = [seq for seq in SeqIO.parse(fi, 'fasta')]
repeats = [seq.id for seq in ref_genome_masked if seq.id not in nuclear_chromosomes]

repeats_alias = {'ref|NC_001224|':'mtDNA',
                 'rDNA.cons_trim':'rDNA',
                 'X_element.cons_trim':'X element',
                 'Y_prime_element.cons_trim':'Y\' element',
                 'TY1-I#LTR/Copia':'Ty1-I',
                 'TY1-LTR#LTR/Copia':'Ty1-LTR',
                 'TY2-I#LTR/Copia':'Ty2-I',
                 'TY2-LTR#LTR/Copia':'Ty2-LTR',
                 'TY3-I#LTR/Gypsy':'Ty3-I',
                 'TY3-LTR#LTR/Gypsy':'Ty3-LTR',
                 'TY3_1p-I#LTR/Gypsy':'Ty3p-I',
                 'TY3_1p-LTR#LTR/Gypsy':'Ty3p-LTR',
                 'TY4-I#LTR/Copia':'Ty4-I',
                 'TY4-LTR#LTR/Copia':'Ty4-LTR',
                 'TY5-I#LTR/Copia':'Ty5-I',
                 'TY5-LTR#LTR/Copia':'Ty5-LTR'               
                }
repeats_order = list(repeats_alias.keys())

In [ ]:
# Phenotypic data
hu_pheno = pd.read_csv(f'{base_dir}tables/hu_pheno.csv').set_index('strain')
hu_ic50 = pd.read_csv('/home/mathieu/postdoc_heasley/experiments/growth_curves/HU_pheno/tables/GC_IC50.csv').set_index('strain', drop=True)
hu_ic50 = hu_ic50.loc[(hu_ic50['dataset']=='wt_log_sat') & (hu_ic50['phase']=='sat')].loc[wine_subclade]

In [ ]:
GC_LOGISTIC = pd.read_csv('/home/mathieu/postdoc_heasley/experiments/growth_curves/HU_pheno/tables/GC_LOGISTIC.csv')
gclog = GC_LOGISTIC.loc[(GC_LOGISTIC['background'].isin(strain_order))
    & (GC_LOGISTIC['dataset']=='wt_ypd_sc')
    & (GC_LOGISTIC['HU']==0)
    & (GC_LOGISTIC['medium']=='YPD')
    & (GC_LOGISTIC['OD']=='log_OD600')].set_index('strain')['mu']

In [ ]:
vcf = allel.read_vcf(f'{base_dir}mpileup_all/mpileup_biallelic_snps.with_parents.add_tags.annot.vcf.gz', fields=['samples', 'calldata/*', 'variants/*'])
vcf_samples = vcf['samples']
GT = allel.GenotypeChunkedArray(vcf['calldata/GT'])

In [ ]:
# Define MAF and segregating sites
#maf = np.nanmax(vcf['calldata/VAF'], axis=2)
ac = GT.count_alleles()
segregating = ac.is_segregating()
#Segregating = np.repeat(np.array(segregating).reshape(-1, 1), 107, axis=1)

# missing genotypes
is_missing = np.array(GT.is_missing())
is_het = np.array(GT.is_het())

In [ ]:
fig, axes = plt.subplots(nrows=3, figsize=[8,8])

for x, ax_idx in zip(['background', 'DW96 COL', 'DW96 ROW'], range(3)):
    ax = axes[ax_idx]
    sns.boxplot(x=x, y='missing', data=strains_info, ax=ax, fliersize=0, color='w')
    sns.stripplot(x=x, y='missing', data=strains_info, ax=ax)
    ax.axhline(1e4)
plt.tight_layout()
plt.show()
plt.close()

# Filters for spores dataset

In [ ]:
F_spores = {}
F_variants = {}
F_segr = {}

for bg, df in strains_info.loc[(strains_info['background'].isin(wine_subclade)) &
    (strains_info['Missing']==False) & 
    (strains_info['strain']!='YJM964_aHyg_1')].groupby('background'):
    
    spores = [s for s in df.index if s != bg]

    f_spores = np.isin(vcf_samples, spores)
    idx_parent = np.where(vcf_samples==bg)[0][0]
    het_parent = is_het[:, idx_parent]
    
    hom_spores = (is_het[:, f_spores].sum(axis=1)==0).flatten()
    gt = GT[:,f_spores]

    f_segr = gt.count_alleles().is_segregating()
    
    f_variants = hom_spores & het_parent & f_segr & ~np.isin(vcf['variants/CHROM'], ['Mito', 'ref|NC_001224|'])
    F_spores[bg] = f_spores
    F_variants[bg] = f_variants
    F_segr[bg] = f_segr

F_variants_all = np.concatenate([a.reshape(-1 ,1) for a in F_variants.values()], axis=1).sum(axis=1)>0
F_spores_all = np.concatenate([a.reshape(-1 ,1) for a in F_spores.values()], axis=1).sum(axis=1)>0

In [ ]:
S = strains_info.loc[(~strains_info['Missing']) & (strains_info['background'].isin(wine_subclade))]\
    .sort_values(by=['background','spore_no'], ascending=[True,True]).index
Sidx = strains_info.loc[S, 'vcf_idx'].values
S_parents = np.isin(S, wine_subclade)

fig, ax = plt.subplots(figsize=[10,15])

sub_gt = GT[F_variants_all][:,Sidx].values.T

dat1 = sub_gt.sum(axis=0) == 1
mask_parents = np.ones(dat1.shape)
mask_parents[S_parents] = 0
ma1 = np.ma.array(dat1, mask=mask_parents)

ax.imshow(ma1, cmap='Greens', aspect='auto', interpolation='nearest')


dat2 = sub_gt.mean(axis=0)
mask_spores = np.zeros(dat2.shape)
mask_spores[S_parents] = 1
ma2 = np.ma.array(dat2, mask=mask_spores)

ax.imshow(ma2, cmap='Blues', vmin=0, vmax=1, aspect='auto', interpolation='nearest')

ax.set_yticks(np.arange(Sidx.shape[0]), labels=S, rotation=0)

chroms_idx = np.array([snpeff_to_ctg[c] for c in vcf['variants/CHROM'][F_variants_all]])

for chrom in nuclear_chromosomes:
    
    x1,x2 = np.where(chroms_idx==chrom)[0][[0, -1]]
    fa = FancyArrow(x1, -3, x2-x1, 0, width=2, fc='w',
                    length_includes_head=True, head_length=15, head_width=2,
                    clip_on=False)
    ax.add_patch(fa)
    ax.text(np.mean([x1,x2]), -3, chr_ctg_to_alias[chrom][3:], ha='center', va='center')
    ax.axvline(x2, c='r', zorder=2)

ax.set_xlim(3900, 4400)
plt.tight_layout()
plt.savefig(f'{fig_path}variable_gt_spores.png', dpi=300)
#plt.show()
plt.close()

In [ ]:
# write geno matrix for r/qtl2

geno_matrix = GT[F_variants_all][:, F_spores_all].values.astype(str)

geno_matrix[geno_matrix=='0'] = 'A'
geno_matrix[geno_matrix=='1'] = 'B'
geno_matrix[geno_matrix=='-1'] = '-'
geno_matrix = np.apply_along_axis(lambda x: f'{x[0]}{x[1]}', 2, geno_matrix)
geno_matrix[geno_matrix=='--'] = '-'
geno_matrix[geno_matrix=='AB'] = '-'

geno_matrix = pd.DataFrame(geno_matrix, columns=vcf_samples[F_spores_all], index=[f'v{i}' for i in range(geno_matrix.shape[0])]).T
#geno_matrix.to_csv(f'{base_dir}rqtl2/geno.csv')

pmap = pd.DataFrame([geno_matrix.columns, 
                     vcf['variants/CHROM'][F_variants_all],
                     vcf['variants/POS'][F_variants_all]*1e-6], # in mega bp
                     index=['marker','chr','pos']).T
#pmap.to_csv(f'{base_dir}rqtl2/pmap.csv', index=False)

gmap = pd.DataFrame([geno_matrix.columns, 
                     vcf['variants/CHROM'][F_variants_all],
                     vcf['variants/POS'][F_variants_all]*1e-3], # cM per kb
                     index=['marker','chr','pos']).T
gmap['pos'] *= gmap['chr'].apply(lambda x: genetic_distance[x])
#gmap.to_csv(f'{base_dir}rqtl2/gmap.csv', index=False)


pheno = hu_pheno.loc[geno_matrix.index, 'inhibition'].rename('HU')
#pheno.to_csv(f'{base_dir}rqtl2/pheno.csv')

In [ ]:
# Correlations with phenotype
S = vcf_samples[F_spores_all]
hu_inhibition = hu_pheno.loc[S, 'inhibition'].values
#sub_gt = GT[F_variants_all][:,F_spores_all].values.mean(axis=2)
GT_PHENO = geno_matrix.copy()
GT_PHENO['inhibition'] = hu_inhibition

In [ ]:
GT_PHENO_SPEARMAN = []
for v in GT_PHENO.columns[:-1]:
    diff = GT_PHENO[[v, 'inhibition']].groupby(v)['inhibition'].median()

    if '--' in diff.index:
        diff = diff.drop('--')
    if diff.shape[0] == 2:
        diff = np.abs((diff*np.array([1, -1])).sum())
        r, p = stats.spearmanr(GT_PHENO[v], GT_PHENO['inhibition'])
    else:
        r, p, diff = np.repeat(np.nan, 3)
    
    GT_PHENO_SPEARMAN.append([v, r, p, diff])
GT_PHENO_SPEARMAN = pd.DataFrame(GT_PHENO_SPEARMAN, columns=['variant_idx', 'r', 'p_value', 'diff'])

In [ ]:
GT_PHENO_SPEARMAN['log_pval'] = (-1)*np.log10(GT_PHENO_SPEARMAN['p_value'])
GT_PHENO_SPEARMAN['chrom'] = vcf['variants/CHROM'][F_variants_all]
GT_PHENO_SPEARMAN['ctg'] = GT_PHENO_SPEARMAN['chrom'].apply(lambda x: snpeff_to_ctg[x])
GT_PHENO_SPEARMAN['pos'] = vcf['variants/POS'][F_variants_all]

In [ ]:
for (chrom, Chrom), df in GT_PHENO_SPEARMAN.groupby(['chrom','Chrom']):
    fig, ax = plt.subplots()
    df = df.copy()
    df['Pos'] = df['pos']/tig_off.loc[chrom, 'len']
    sns.scatterplot(x='diff', y='log_pval', hue='Pos', 
                    palette='rainbow', hue_norm=(0,1), data=df, ax=ax)
    ax.set_title(Chrom)
    ax.set_xlim(0, 0.3)
    ax.set_ylim(0, 5)
    plt.show()
    plt.close()

# QTL analysis from R

In [ ]:
lod = pd.read_csv(f'{base_dir}rqtl2/lod.csv', index_col=0)
lod_peaks = pd.read_csv(f'{base_dir}rqtl2/peaks.csv', index_col=0)

lod = pd.concat([lod, GT_PHENO_SPEARMAN.set_index('variant_idx')[['chrom','ctg','pos']]], axis=1)
lod['Pos'] = lod['pos'] + tig_off.loc[lod['ctg'], 'cumul_start'].values
lod['color'] = lod['ctg'].apply(lambda x: chr_ctg_color[x])

In [ ]:
f = np.isin(geno_matrix.columns, lod.loc[lod['HU']>2].index)
qtl_hits = vcf['variants/ANN'][F_variants_all][f]
qtl_hits = pd.DataFrame([l.split('|') for l in qtl_hits])
qtl_hits.columns = ['Allele', 'Annotation', 'Annotation_Impact',
                    'Gene_Name', 'Gene_ID', 'Feature_Type',
                    'Feature_ID', 'Transcript_BioType', 'Rank',
                    'HGVS.c', 'HGVS.p', 'cDNA.pos / cDNA.length',
                    'CDS.pos / CDS.length', 'AA.pos / AA.length', 'Distance', 
                    'ERRORS / WARNINGS / INFO']

qtl_hits = pd.concat([qtl_hits, lod.loc[f].reset_index()], axis=1)

In [ ]:
# export QTL hits
#qtl_hits.sort_values(by='Pos').to_excel(f'{base_dir}tables/qtl_hits.xlsx', index=False)

## Fig S6A

In [ ]:
fig, ax = plt.subplots(figsize=[10,4])

ax.scatter(lod['Pos']*1e-6, lod['HU'], color=lod['color'], s=3, zorder=1)

for chrom in nuclear_chromosomes:
    
    #x1,x2 = np.where(chroms_idx==chrom)[0][[0, -1]]
    x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']] * 1e-6
    #x2 = x1 + l
    
    fa = FancyArrow(x1, -0.5, x2-x1, 0, width=0.3, fc='w', length_includes_head=True,
                    head_length=0, head_width=0.3, clip_on=False)
    ax.add_patch(fa)
    ax.text(np.mean([x1,x2]), -0.5, chr_ctg_to_alias[chrom][3:], ha='center', va='center')
    
    for y in (2,3):
        ax.axhline(y, c='k', lw=1, ls='--', zorder=0)

ax.set_ylabel('LOD score')
ax.set_xlabel('position (Mb)')
ax.set_xmargin(0.02)

plt.tight_layout()
sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS6A.{ext}', dpi=300)

plt.show()
plt.close()

## Genome-wide heterozygosity levels

In [ ]:
fig, ax = plt.subplots()
for i,s in enumerate(vcf['samples']):
    if 'YJM947' in s:
        f = is_het[:,i]
        chrom = vcf['variants/CHROM']
        pos = vcf['variants/POS'] + tig_off.loc[chrom, 'cumul_start'].values

        ax.plot(pos[f], np.arange(f.sum()))

# Depth

In [ ]:
# Import depth
DEPTH_NUC = []
idx = 0

with ProgressBar(max_value=strains_info.shape[0]) as bar:
    
    for s in strains_info.index:
        if strains_info.loc[s, 'ho'] == 'WT':
            depth_path = f'{base_dir}../short_read_project/data/depth/{s}.depth_nuc_bin.tab.gz'
        else:
            depth_path = f'{base_dir}data/depth/{s}.depth_nuc_bin.tab.gz'
        depth = pd.read_csv(depth_path, header=None, sep='\t')
        depth.columns = ['CHROM', 'POS', 'DP']
        depth['dp_rel'] = depth['DP']/depth.loc[depth['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
        
        depth['strain'] = s

        DEPTH_NUC.append(depth)

        idx += 1
        bar.update(idx)

DEPTH_NUC = pd.concat(DEPTH_NUC).reset_index(drop=True)
for c in ['background', 'ho', 'MAT', 'DW96 ROW', 'DW96 COL']:
    DEPTH_NUC[c] = strains_info.loc[DEPTH_NUC['strain'], c].values

In [ ]:
S = strains_info.loc[(~strains_info['Missing']) & (strains_info['background'].isin(wine_subclade))]\
    .sort_values(by=['background','spore_no'], ascending=[True,True]).index
dat = DEPTH_NUC.loc[DEPTH_NUC['CHROM']=='ref|NC_001140|'].pivot_table(index='strain', columns='POS', values='dp_rel')
fig, ax = plt.subplots(figsize=[4,20])
sns.heatmap(dat.loc[S], vmin=0, vmax=3, cbar=False, ax=ax)
plt.tight_layout()
plt.savefig(f'{fig_path}geno_transloc.png', dpi=300)
#plt.show()
plt.close()

# Repeats CN

In [ ]:
DP_REP = []
for s, df in DEPTH_NUC.groupby('strain'):
    dp_nuc = df.loc[df['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
    df = df.loc[df['CHROM'].isin(repeats)]
    added_repeat = {r:False for r in repeats}
    for r, df1 in df.groupby('CHROM'):
        DP_REP.append([s, r, df1['DP'].median(), dp_nuc])
        added_repeat[r] = True
    for r,a in added_repeat.items():
        if a == False:
            DP_REP.append([s, r, 0, dp_nuc])
DP_REP = pd.DataFrame(DP_REP, columns=['strain', 'repeat', 'depth', 'depth_nuc'])
DP_REP['cn'] = DP_REP['depth']/DP_REP['depth_nuc']
for c in ['background', 'ho', 'MAT', 'DW96 ROW', 'DW96 COL']:
    DP_REP[c] = strains_info.loc[DP_REP['strain'], c].values

In [ ]:
#DP_REP.to_csv(f'{base_dir}tables/repeats_cn.tsv', sep='\t', index=None)
DP_REP = pd.read_csv(f'{base_dir}tables/repeats_cn.tsv', sep='\t')

## Fig S5A

In [ ]:
fig, ax = plt.subplots(figsize=[6,5])

dat = DP_REP.loc[(DP_REP['repeat']=='Y_prime_element.cons_trim') 
    & (DP_REP['background']!='YJM434') & (DP_REP['DW96 ROW']!='D')]

sns.boxplot(x='background', y='cn', order=strain_order, ax=ax, 
            color='k', fill=False, linewidth=1,
            fliersize=0,
            data=dat.loc[(dat['ho']!='WT')])

sns.stripplot(x='background', y='cn', order=strain_order, ax=ax, 
              hue='background', palette=strain_yprime_color,
              s=6, edgecolor='k', linewidth=0.5,
              data=dat.loc[(dat['ho']!='WT')])

ax.bar(np.arange(10), dat.set_index('strain').loc[strain_order, 'cn'], 
       width=1, color=[strain_yprime_color[s] for s in strain_order], alpha=0.7, zorder=0)

for x, s in enumerate(strain_order):

    n_spores = dat.set_index('background').loc[s].shape[0]-1
    ax.text(x, 230, f'({n_spores})', size=9, ha='center', va='center')
    
ax.set_ylabel('Y\' CN')
ax.set_xlabel('')
ax.tick_params('x', rotation=90)

sns.despine()
plt.tight_layout()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS5A.{ext}', dpi=300)

plt.show()
plt.close()

## Fig S5B

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=[11,5])

strain_axes = dict(zip(strain_order, itertools.product(range(2), range(5))))

dat = pd.merge(hu_pheno.drop(['background', 'MAT'], axis=1).reset_index(),
               DP_REP.loc[(DP_REP['repeat']=='Y_prime_element.cons_trim') &
                   (DP_REP['DW96 ROW']!='D')], on='strain')

for bg, df in dat.sort_values(by='cn').groupby('background'):
    ax = axes[strain_axes[bg]]
    X = df['cn'].iloc[[0, -1]]
    c = strain_yprime_color[bg]
    ax.scatter(df['cn'], df['inhibition']*100, color=c, zorder=2)

    lm = stats.linregress(df['cn'], df['inhibition'])
    ax.plot(X, (X*lm.slope+lm.intercept)*100, c='k', zorder=1)

    ax.set_title(f'{bg}\n'+'r$^2$'+f'={lm.rvalue**2:.2f} p={mh.plot_pval_text(lm.pvalue)}')
    ax.set_ylabel('HU 128 mM inhib (%)')
    ax.set_xlabel('Y\' CN')

sns.despine()
plt.tight_layout()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS5B.{ext}', dpi=300)
    
plt.show()
plt.close()

## Fig S6B

In [ ]:
variants_sub = ['v582', 'v1332', 'v1355']
fig, axes = plt.subplots(ncols=5, figsize=[10,4])

for ax_idx, v in enumerate(variants_sub):
    ax = axes[ax_idx]
    dat = GT_PHENO[[v,'inhibition']]
    dat = dat.loc[dat[v]!='-']
    dat['gt'] = dat[v].replace({'AA':'ref', 'BB':'alt'})
    dat['background'] = strains_info.loc[dat.index, 'background']
    dat['inhibition'] *= 100
    
    sns.boxplot(x='gt', y='inhibition', order=['ref', 'alt'], 
               color='w', linecolor='k', fliersize=0, 
               data=dat, ax=ax)
    sns.swarmplot(x='gt', y='inhibition',  order=['ref', 'alt'],
               hue='background', palette=strain_yprime_color, dodge=False,
               data=dat, ax=ax, legend=False)

    chrom, pos, gene_name, hgvs_c, annot_impact = qtl_hits.set_index('index').loc[v, ['chrom', 'pos', 'Gene_Name', 'HGVS.c', 'Annotation_Impact']]
    name = f'{chrom}:{pos}\n{gene_name}\n{hgvs_c}\n{annot_impact}'
    ax.set_xlabel(name)

dat = pd.concat([hu_pheno, strains_info[['translocation', 'disomy8']]], axis=1)
dat = dat.loc[~dat['translocation'].isna()]
dat['gt_translocation'] = dat['translocation'].replace({0:'false', 1:'true'})
dat['gt_disomy8'] = dat['disomy8'].replace({0:'false', 1:'true'})
dat['background'] = strains_info.loc[dat.index, 'background']
dat['inhibition'] *= 100

ax = axes[3]
sns.boxplot(x='gt_translocation', y='inhibition', order=['false', 'true'],
               color='w', linecolor='k',
               data=dat, ax=ax)
sns.swarmplot(x='gt_translocation', y='inhibition', order=['false', 'true'],
           hue='background', palette=strain_yprime_color, dodge=False,
           data=dat, ax=ax, legend=False)
ax.set_xlabel('VIII-XVI translocation')

ax = axes[4]
sns.boxplot(x='gt_disomy8', y='inhibition', order=['false', 'true'],
               color='w', linecolor='k',
               data=dat, ax=ax)
sns.swarmplot(x='gt_disomy8', y='inhibition', order=['false', 'true'],
           hue='background', palette=strain_yprime_color, dodge=False,
           data=dat, ax=ax, legend=False)
ax.set_xlabel('VIII disomy')

for ax in axes:
    ax.set_ylabel('HU 128 mM inhib (%)')

sns.despine()
plt.tight_layout()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS6B.{ext}', dpi=300)
    
plt.show()
plt.close()

In [ ]:
dat = pd.concat([hu_pheno, strains_info[['translocation', 'disomy8']]], axis=1)
dat = dat.loc[~dat['disomy8'].isna()]
u,v = dat.groupby('disomy8')['inhibition']
stats.ttest_ind(u[1], v[1])

# GWAS for parents

## Filters for parents dataset

In [ ]:
vcf_parents = allel.read_vcf(f'{base_dir}gwas/wine_subclade.biallelic.vcf.gz', fields=['samples', 'calldata/*', 'variants/*'])
vcf_parents_samples = vcf_parents['samples']
GT_parents = allel.GenotypeChunkedArray(vcf_parents['calldata/GT'])

In [ ]:
F_segr_parents =(np.var(GT_parents.to_n_alt(), axis=1)>0) \
& np.any(GT_parents.is_het() & ((vcf_parents['calldata/VAF1']>0.3) | (vcf_parents['calldata/VAF1']<0.7)), axis=1) \
& (vcf_parents['calldata/AD'][:,:,:2].reshape(GT_parents.shape[0], -1).max(axis=1) > 20) \
& ~np.isin(vcf_parents['variants/CHROM'], ['Mito', 'ref|NC_001224|']) \
& ~np.any(GT_parents.is_missing(), axis=1)


with open(f'{base_dir}gwas/F_segr_parents.txt', 'w') as handle:
    handle.write('\n'.join(F_segr_parents.astype(str)))

In [ ]:
S = vcf_samples[F_parents]
sub_gt = GT_parents[F_segr_parents].to_n_alt()

GT_PHENO_PARENTS = pd.DataFrame(sub_gt.T, index=S)
GT_PHENO_PARENTS['IC50'] = hu_ic50.loc[S, 'IC50']
GT_PHENO_PARENTS['mu'] = gclog.loc[S]
GT_PHENO_PARENTS['strain'] = S

GT_PHENO_PARENTS_CORR = []
for i in np.arange(F_segr_parents.sum()):
    
    r, p = stats.pearsonr(GT_PHENO_PARENTS[i], GT_PHENO_PARENTS['mu'])
        
    GT_PHENO_PARENTS_CORR.append([i, r, p])
GT_PHENO_PARENTS_CORR = pd.DataFrame(GT_PHENO_PARENTS_CORR, columns=['variant_idx', 'r', 'pvalue']).set_index('variant_idx')


GT_PHENO_PARENTS_CORR['chrom'] = vcf_parents['variants/CHROM'][F_segr_parents]
GT_PHENO_PARENTS_CORR['ctg'] = GT_PHENO_PARENTS_CORR['chrom'].apply(lambda x: snpeff_to_ctg[x])
GT_PHENO_PARENTS_CORR['pos'] = vcf_parents['variants/POS'][F_segr_parents]
GT_PHENO_PARENTS_CORR['Pos'] = GT_PHENO_PARENTS_CORR['pos'] + tig_off.loc[GT_PHENO_PARENTS_CORR['ctg'], 'cumul_start'].values
GT_PHENO_PARENTS_CORR = GT_PHENO_PARENTS_CORR.loc[~GT_PHENO_PARENTS_CORR['pvalue'].isna()]
GT_PHENO_PARENTS_CORR['log_pvalue'] = (-1)*np.log10(GT_PHENO_PARENTS_CORR['pvalue'])
GT_PHENO_PARENTS_CORR['padj'] = multipletests(GT_PHENO_PARENTS_CORR['pvalue'], method='fdr_bh')[1]
GT_PHENO_PARENTS_CORR['log_padj'] = (-1)*np.log10(GT_PHENO_PARENTS_CORR['padj'])

parents_hits = vcf_parents['variants/ANN'][F_segr_parents]
parents_hits = pd.DataFrame([l.split('|') for l in parents_hits])
parents_hits.columns = ['Allele', 'Annotation', 'Annotation_Impact',
                    'Gene_Name', 'Gene_ID', 'Feature_Type',
                    'Feature_ID', 'Transcript_BioType', 'Rank',
                    'HGVS.c', 'HGVS.p', 'cDNA.pos / cDNA.length',
                    'CDS.pos / CDS.length', 'AA.pos / AA.length', 'Distance', 
                    'ERRORS / WARNINGS / INFO']

GT_PHENO_PARENTS_CORR = GT_PHENO_PARENTS_CORR.merge(parents_hits, left_index=True, right_index=True)

# rrBLUP GWAS

In [ ]:
random_gwas = pd.read_csv(f'{base_dir}gwas/random.rrblup_gwas.csv', index_col=0)
gwas_threshold = np.quantile(random_gwas.groupby('iter')['mu'].max(), 0.95)

In [ ]:
plt.plot(np.sort(random_gwas.groupby('iter')['mu'].max()), np.linspace(0, 100, 500))
plt.axhline(95)
plt.axvline(gwas_threshold)

In [ ]:
gwas = pd.read_csv(f'{base_dir}gwas/rrblup_gwas.csv', index_col=0)
gwas.columns = [f'gwas.{c}' for c in gwas.columns]
GWAS = pd.concat([GT_PHENO_PARENTS_CORR, gwas.reset_index(drop=True)], axis=1)
GWAS['Pos'] = GWAS['pos']+tig_off.loc[GWAS['ctg'], 'cumul_start'].values
GWAS['color'] = GWAS['ctg'].apply(lambda x: chr_ctg_color[x])

## Fig 3B

In [ ]:
fig, ax = plt.subplots(figsize=[8,4],
                        gridspec_kw=dict(wspace=0.3, 
                                         left=0.12, top=0.9, right=0.97, bottom=0.15))

ax.scatter(GWAS['Pos']*1e-6, GWAS['gwas.mu'], color='k', s=3, zorder=2)

for chrom in nuclear_chromosomes:
    
    #x1,x2 = np.where(chroms_idx==chrom)[0][[0, -1]]
    x1, x2 = tig_off.loc[chrom, ['cumul_start', 'cumul_end']] * 1e-6
    #x2 = x1 + l
    
    fa = FancyArrow(x1, -0.5, x2-x1, 0, width=0.3, fc='w', length_includes_head=True,
                    head_length=0, head_width=0.3, clip_on=False)
    ax.add_patch(fa)
    ax.text(np.mean([x1,x2]), -0.5, chr_ctg_to_alias[chrom][3:], ha='center', va='center')
    
    bg_c = chr_ctg_color[chrom]

    fa = FancyArrow(x1, 1.5, x2-x1, 0, width=5, fc=bg_c, length_includes_head=True,
                linewidth=0, head_length=0, head_width=0.3, clip_on=True, 
                    alpha=0.25, zorder=0)
    ax.add_patch(fa)
    
ax.axhline(gwas_threshold, ls='--', lw=1, c='k', zorder=1)

ax.set_yticks(range(0,5))
ax.set_ylabel('GWAS score\n(-log$_{10}$ p-value)')
ax.set_xlabel('position (Mb)')

ax.set_xmargin(0.01)

fig.text(0.02, 0.93, 'B', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig3B.{ext}', dpi=300)

plt.show()
plt.close()